In [39]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from keras import initializers
import ast

Datasets = []
PREDICTORS = ["PwmD", "PwmE"]   
TARGET_INT = ["Theta"]  
TARGET = ["DeltaTheta"]
SETS = [
    "Train_1",
    "Train_2",
    "Test_1",
    "Test_2",
    "Test_3",
    "Val",
    "LSG1",
    "LSG2"
]
     
TIME_STEPS = 3
TS = 0.07

In [40]:
for i in range(4):
    Dataset = pd.read_excel(f"./../../RotedData/Data.xlsx", f"D{i+1}")   
    Datasets.append(Dataset)

for i in range(4):   
    Dataset = pd.read_csv(f"./../../Data/Data{i + 1}.csv")  
    Datasets.append(Dataset)
    
    
for i in range(len(Datasets)):
    Dataset = Datasets[i].copy()

    for var in TARGET_INT:
        Dataset[f"Delta{var}"] = (Dataset[var].shift(-1) - Dataset[var]) / TS

    Dataset = Dataset.dropna(subset=[f"Delta{var}" for var in TARGET_INT])

    Datasets[i] = Dataset

In [41]:
results = [] 
for target in TARGET_INT:
    result = pd.read_excel(f"./models/{target}.xlsx")
    results.append(result)

In [57]:
import joblib

TARGET_PATH = f"./Re-Run/Theta/Ts-03"
SCALER = joblib.load(f"{TARGET_PATH}/scalers/scaler.pkl")
OUT_SCALER = joblib.load(f"{TARGET_PATH}/scalers/out_scaler.pkl")

NormDatasets = []

Train1 = Datasets[0].copy()
Train1[PREDICTORS] = SCALER.fit_transform(Train1[PREDICTORS])
Train1[TARGET] = OUT_SCALER.fit_transform(Train1[TARGET])
NormDatasets.append(Train1)

for i in range(7):
    CurrentTestDataset = Datasets[i + 1].copy()
    CurrentTestDataset[PREDICTORS] = SCALER.transform(CurrentTestDataset[PREDICTORS])
    CurrentTestDataset[TARGET] = OUT_SCALER.transform(CurrentTestDataset[TARGET])
    NormDatasets.append(CurrentTestDataset)

In [44]:
import pandas as pd

def PickModels(df, target, tr1=0.0, tr2=0.0, v1=0.0, t3=0.0):

    r2_tr1 = f"R2_Train_1_{target}"
    r2_tr2 = f"R2_Train_2_{target}"
    r2_val = f"R2_Val_{target}"
    r2_t3 = f"R2_Test_3_{target}"

    # filtro mínimo
    filtered = df[
        (df[r2_tr1] > tr1) &
        (df[r2_tr2] > tr2) &
        (df[r2_t3] > t3) &
        (df[r2_val] > v1)
    ]

    if filtered.empty:
        print("Nenhum modelo satisfaz os critérios.")
        return None

    # 🔹 melhor para cada métrica
    best_tr1 = filtered.sort_values(r2_tr1, ascending=False).iloc[0]
    best_tr2 = filtered.sort_values(r2_tr2, ascending=False).iloc[0]
    best_val = filtered.sort_values(r2_val, ascending=False).iloc[0]
    best_t3 = filtered.sort_values(r2_t3, ascending=False).iloc[0]

    selected = pd.DataFrame([best_tr1, best_tr2, best_val, best_t3]).drop_duplicates()

    cols = ["model", "Neurons", "reg", "seed", r2_tr1, r2_tr2, r2_val, r2_t3]

    table = selected[cols].copy()

    # 🔹 calcular média das métricas
    mean_values = table[[r2_tr1, r2_tr2, r2_val, r2_t3]].mean()

    mean_row = pd.DataFrame([{
        "model": "MEAN",
        "Neurons": "-",
        "reg": "-",
        "seed": "-",
        r2_tr1: mean_values[r2_tr1],
        r2_tr2: mean_values[r2_tr2],
        r2_val: mean_values[r2_val],
        r2_t3: mean_values[r2_t3]
    }])

    table_mean = pd.concat([table, mean_row], ignore_index=True)

    display(table_mean)

    return table

In [45]:
models = []
models.append(PickModels(results[0], "Theta", tr1=0.8, tr2=0.6, v1=0.6, t3=0.6))

,model,Neurons,reg,seed,R2_Train_1_Theta,R2_Train_2_Theta,R2_Val_Theta,R2_Test_3_Theta
0,model_arch16-8-4_r0.01_seed8716,"[16, 8, 4]",0.01,8716,0.901765,0.69352,0.751616,0.773393
1,MEAN,-,-,-,0.901765,0.69352,0.751616,0.773393


In [46]:
from keras.saving import register_keras_serializable
import tensorflow as tf

@register_keras_serializable()
class PINN(tf.keras.Model):

    def __init__(self, architecture, initializer, regularizer):
        super(PINN, self).__init__()

        self.rnn_layers = []

        for i, units in enumerate(architecture):

            if i < len(architecture) - 1:
                self.rnn_layers.append(
                    tf.keras.layers.SimpleRNN(
                        units,
                        activation='tanh',
                        return_sequences=True,
                        kernel_initializer=initializer,
                        kernel_regularizer=regularizer,
                        recurrent_regularizer=regularizer,
                        bias_regularizer=regularizer
                    )
                )
            else:
                self.rnn_layers.append(
                    tf.keras.layers.SimpleRNN(
                        units,
                        activation='tanh',
                        kernel_initializer=initializer,
                        kernel_regularizer=regularizer,
                        recurrent_regularizer=regularizer,
                        bias_regularizer=regularizer
                    )
                )

        # camada densa final
        self.out_layer = tf.keras.layers.Dense(
            1,
            activation="linear",
            kernel_initializer=initializer,
            kernel_regularizer=regularizer,
            bias_regularizer=regularizer
        )

    def call(self, inputs):

        x = inputs

        for rnn in self.rnn_layers:
            x = rnn(x)

        return self.out_layer(x)

In [47]:
import tensorflow as tf
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error

def CreateSequences(input_data, target_data, timesteps):

    X_seq, Y_seq = [], []

    for i in range(timesteps, len(input_data)):
        X_seq.append(input_data.iloc[i-timesteps:i].values)
        Y_seq.append(target_data.iloc[i].values)

    return np.array(X_seq), np.array(Y_seq)


class EnsembleNets():

    def __init__(self, predictors, targets, int_targets, out_scaler):

        self.models = []
        self.predictors = predictors
        self.targets = targets
        self.target_int = int_targets
        self.input_size = len(predictors)
        self.output_size = len(targets)
        self.out_scaler = out_scaler


    def SetModels(self, Wf, architecture, r, s):
        regularizer = tf.keras.regularizers.l2(r)
        init = initializers.RandomNormal(seed=int(s))
        
        model = PINN(
            architecture=architecture,
            initializer=init,
            regularizer=regularizer
        )

        model.build((None, TIME_STEPS, len(PREDICTORS)))
        model.load_weights(Wf)
        self.models.append(model)


    def BuildEnsemble(self, ModelsList):
        for (arch, Wf, r, s) in ModelsList:
            if isinstance(arch, str):
                arch = ast.literal_eval(arch)      
            self.SetModels(Wf, arch, r, s)


    def PredictModel(self, model, x):

        y_pred = model.predict(x)
        y_pred = self.out_scaler.inverse_transform(y_pred)
        init_vals = np.array([Datasets[i][name].iloc[0] for name in self.target_int])
        
        for j in range(self.output_size):
            y_pred[:, j] = init_vals[j] + np.cumsum(y_pred[:, j] * TS)
        return y_pred


    def EvalEnsemble(self, datasets, Wensemble):

        for d, dataset in enumerate(datasets):

            x, y = CreateSequences(
                dataset[self.predictors],
                dataset[self.targets],
                TIME_STEPS
            )

            y_true = dataset[self.target_int].iloc[TIME_STEPS:].values

            y_ensemble = np.zeros((len(x), self.output_size))

            print(f"\n===== DATASET: {SETS[d]} =====")

            # 🔹 métricas individuais
            for i, model in enumerate(self.models):

                y_pred = self.PredictModel(model, x)

                y_ensemble += Wensemble[i] * y_pred

                for j, name in enumerate(self.target_int):

                    r2 = r2_score(y_true[:, j], y_pred[:, j])
                    mse = mean_squared_error(y_true[:, j], y_pred[:, j])

                    print(
                        f"Model {i+1} | {name} -> "
                        f"R² = {r2:.4f}, MSE = {mse:.4e}"
                    )

            # 🔹 métricas do ensemble
            print("---- ENSEMBLE ----")

            for j, name in enumerate(self.target_int):

                r2 = r2_score(y_true[:, j], y_ensemble[:, j])
                mse = mean_squared_error(y_true[:, j], y_ensemble[:, j])

                print(
                    f"Ensemble | {name} -> "
                    f"R² = {r2:.4f}, MSE = {mse:.4e}"
                )

In [48]:
def Test(predictors, targets, int_targets, out_scaler, i):
    ModelsList = []

    for _, row in models[i].iterrows():

        # garante que arch é lista
        arch = row["Neurons"]
        r = float(row["reg"])
        Wf = f".\Re-Run\{int_targets[0]}\Ts-03\weights\{row['model']}.weights.h5"
        s = float(row["seed"])
        ModelsList.append((arch, Wf, r, s))
        
    Wensemble = np.ones(len(ModelsList)) / len(ModelsList) 
    
    ensemble = EnsembleNets(
        predictors=predictors,
        targets=targets,
        int_targets=int_targets,
        out_scaler=out_scaler,
    )

    ensemble.BuildEnsemble(ModelsList)

    ensemble.EvalEnsemble(
        datasets=NormDatasets,
        Wensemble=Wensemble,
    )

In [50]:
Test(predictors=["PwmD", "PwmE"],
     targets=["DeltaTheta"],
     int_targets=["Theta"],
     out_scaler=OUT_SCALER, 
     i=0)


===== DATASET: Train_1 =====
31/31 [==============================] - 0s 2ms/step
Model 1 | Theta -> R² = 0.8040, MSE = 4.4607e-02
---- ENSEMBLE ----
Ensemble | Theta -> R² = 0.8040, MSE = 4.4607e-02

===== DATASET: Train_2 =====
31/31 [==============================] - 0s 2ms/step
Model 1 | Theta -> R² = -3.9223, MSE = 1.1237e+00
---- ENSEMBLE ----
Ensemble | Theta -> R² = -3.9223, MSE = 1.1237e+00

===== DATASET: Test_1 =====
31/31 [==============================] - 0s 2ms/step
Model 1 | Theta -> R² = 0.4440, MSE = 1.2738e-01
---- ENSEMBLE ----
Ensemble | Theta -> R² = 0.4440, MSE = 1.2738e-01

===== DATASET: Test_2 =====
31/31 [==============================] - 0s 2ms/step
Model 1 | Theta -> R² = -1.5769, MSE = 6.1105e-01
---- ENSEMBLE ----
Ensemble | Theta -> R² = -1.5769, MSE = 6.1105e-01

===== DATASET: Test_3 =====
45/45 [==============================] - 0s 2ms/step
Model 1 | Theta -> R² = 0.6429, MSE = 1.1712e-01
---- ENSEMBLE ----
Ensemble | Theta -> R² = 0.6429, MSE = 1.171